 Privacy-Aware Android App Permission Management
 Borda-Garrett Ranking + Association Rule Mining + Reinforcement Learning

**Category focus:** Social & Communication


## 2. Imports and Configuration

In [1]:
import warnings
warnings.filterwarnings("default")  # show warnings, never hide them

import os
import json
import random
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


In [2]:
# ---------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ---------------------------------------------------------------
# Project configuration (all important parameters live here)
# ---------------------------------------------------------------
CATEGORY = "Social & Communication"
# The raw dataset's `Category` column does not contain a literal
# "Social & Communication" value. It contains the finer-grained
# categories "Communication" and "Social" separately. We map both
# of these raw categories onto the broad project category below.
# This mapping is applied explicitly (Section 6) and is NOT a
# silent fabrication of data -- it is documented here and printed.
CATEGORY_RAW_VALUES = ["Communication", "Social"]

DATA_PATH = "Android_Permission.csv"
WEIGHTS_PATH = "combined_weights_social_communication.csv"
OUTPUT_DIR = "results_RL_Social_and_Communication"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Top-K category-specific permissions used for the RL state/action space.
# Configurable per Step 5 of the specification. NOTE: this only controls what the RL
# agent can Allow/Deny -- the malicious-risk classifier (Section 12) uses a separate,
# broader feature set (CLF_PERMISSION_FEATURES), decoupled from this value.
TOP_K = 12

# Extra real per-app metadata features added to the malicious-risk classifier ONLY
# (never part of the RL action space -- you cannot "Allow/Deny" a rating).
# Section 7b appends two more engineered features (PermissionCount, DangerousToSafeRatio)
# to this list once they are computed.
METADATA_FEATURES = ["Rating", "Number of ratings", "Price",
                     "Dangerous permissions count", "Safe permissions count"]

# SMOTE oversampling target ratio for the minority class in TRAINING data only.
# 1.0 = fully balance minority to match majority count.
SMOTE_TARGET_RATIO = 1.0

# Number of folds for cross-validated classifier model/hyperparameter selection (Section 12).
N_CV_FOLDS = 5

# Association rule mining thresholds (Step 6). These are NOT relaxed
# just to manufacture more rules -- if few/no rules survive, that is
# reported honestly rather than hidden.
MIN_SUPPORT = 0.04
MIN_CONFIDENCE = 0.60
MIN_LIFT = 1.02

# Reward coefficient candidates evaluated on the VALIDATION set (Step 12/16).
# Widened from v1 to include settings where BETA/GAMMA (privacy penalty) are large
# relative to ALPHA (utility reward) -- v1's candidates were all utility-dominant, which is
# part of why the learned policy converged to an ~allow-everything policy indistinguishable
# from the AllowAll baseline. Including higher-penalty candidates lets the validation-based
# selection actually choose a more risk-sensitive policy if one performs better.
COEF_CANDIDATES = [
    (0.60, 0.25, 0.15),
    (0.70, 0.20, 0.10),
    (0.50, 0.30, 0.20),
    (0.45, 0.40, 0.15),
    (0.35, 0.50, 0.15),
    (0.30, 0.55, 0.15),
]

# Q-learning hyperparameters (Step 15)
QL_ALPHA = 0.1           # learning rate
QL_GAMMA = 0.9            # discount factor
QL_EPISODES = 5000
QL_EPS_START = 1.0
QL_EPS_MIN = 0.05
QL_EPS_DECAY = 0.999

# Predefined privacy-score thresholds (Step 19). These are declared as
# FIXED / predefined thresholds, not learned from data.
PRIVACY_LOW_THRESHOLD = 0.33
PRIVACY_HIGH_THRESHOLD = 0.67

print("Configuration loaded.")
print("CATEGORY:", CATEGORY, "<- raw Category values:", CATEGORY_RAW_VALUES)
print("TOP_K:", TOP_K)
print("N_CV_FOLDS:", N_CV_FOLDS)
print("Association rule thresholds:", MIN_SUPPORT, MIN_CONFIDENCE, MIN_LIFT)
print("Reward coefficient candidates:", COEF_CANDIDATES)


Configuration loaded.
CATEGORY: Social & Communication <- raw Category values: ['Communication', 'Social']
TOP_K: 12
N_CV_FOLDS: 5
Association rule thresholds: 0.04 0.6 1.02
Reward coefficient candidates: [(0.6, 0.25, 0.15), (0.7, 0.2, 0.1), (0.5, 0.3, 0.2), (0.45, 0.4, 0.15), (0.35, 0.5, 0.15), (0.3, 0.55, 0.15)]


## 3. Load Dataset

Loads the raw Android permission dataset and performs basic structural checks
(shape, missing values, class balance, category distribution).

In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
print()
print("Missing values per column (only columns with >0 missing shown):")
missing_counts = df.isna().sum()
print(missing_counts[missing_counts > 0])
print()
print("Class distribution (0 = Benign, 1 = Malicious):")
print(df["Class"].value_counts())
print()
print("Category distribution (top 15):")
print(df["Category"].value_counts().head(15))

Dataset shape: (29999, 184)

Missing values per column (only columns with >0 missing shown):
App                              1
Description                      3
Related apps                   755
Dangerous permissions count    204
dtype: int64

Class distribution (0 = Benign, 1 = Malicious):
Class
1    20000
0     9999
Name: count, dtype: int64

Category distribution (top 15):
Category
Entertainment        2827
Travel & Local       2154
Arcade & Action      1959
Books & Reference    1959
Brain & Puzzle       1873
Casual               1678
Personalization      1527
Lifestyle            1435
Tools                1329
Education            1305
Communication        1103
Finance              1045
Health & Fitness     1038
Business              945
Cards & Casino        881
Name: count, dtype: int64


## 4. Load Combined Borda-Garrett Weights (Precomputed Project Input)

`combined_weights_social_communication.csv` contains the **already-computed** Borda + Garrett
combined importance weight for each permission in this category. These weights were derived
externally (outside this notebook) from the full dataset methodology described in the project,
and are treated here strictly as a **precomputed input** -- they are loaded, validated, and used,
but **never recomputed**.

In [4]:
weights_df = pd.read_csv(WEIGHTS_PATH)
print("Raw combined weights:")
print(weights_df)

weight_sum = weights_df["CombinedWeight"].sum()
print("\nSum of CombinedWeight:", weight_sum)

if not np.isclose(weight_sum, 1.0, atol=1e-2):
    print("Sum is not ~1.0 -> normalizing.")
    weights_df["CombinedWeight"] = weights_df["CombinedWeight"] / weight_sum
else:
    print("Sum is already ~1.0 -> no normalization needed.")

weights_df = weights_df.sort_values("CombinedWeight", ascending=False).reset_index(drop=True)
weights_df["Rank"] = weights_df.index + 1

weight_map = dict(zip(weights_df["Permission"], weights_df["CombinedWeight"]))

print("\nCombined Borda + Garrett weights (sorted descending):")
weights_df

Raw combined weights:
          Permission  CombinedWeight
0           Location        0.105413
1           Internet        0.105413
2            Storage        0.104215
3             Camera        0.100210
4         PhoneState        0.100058
5         Microphone        0.097402
6          Bluetooth        0.088193
7           Calendar        0.084035
8           Contacts        0.080030
9                SMS        0.080030
10          CallLogs        0.027501
11  ReadPhoneNumbers        0.027501

Sum of CombinedWeight: 0.9999999999999996
Sum is already ~1.0 -> no normalization needed.

Combined Borda + Garrett weights (sorted descending):


,Permission,CombinedWeight,Rank
0,Location,0.105413,1
1,Internet,0.105413,2
2,Storage,0.104215,3
3,Camera,0.100210,4
4,PhoneState,0.100058,5
5,Microphone,0.097402,6
6,Bluetooth,0.088193,7
7,Calendar,0.084035,8
8,Contacts,0.080030,9
9,SMS,0.080030,10


## 5. Data Validation

In [5]:
PERMISSION_COLUMN_MAP = {
    "Camera": [
        "Hardware controls : take pictures and videos (D)"
    ],
    "Location": [
        "Your location : coarse (network-based) location (D)",
        "Your location : fine (GPS) location (D)",
        "Default : coarse (network-based) location (S)"
    ],
    "Contacts": [
        "Your personal information : read contact data (D)",
        "Your personal information : write contact data (D)",
        "Default : write contact data (S)",
        "Your accounts : contacts data in Google accounts (D)"
    ],
    "Microphone": [
        "Hardware controls : record audio (D)"
    ],
    "Storage": [
        "Storage : modify/delete USB storage contents modify/delete SD card contents (D)",
        "Default : modify/delete USB storage contents modify/delete SD card contents (S)"
    ],
    "SMS": [
        "Services that cost you money : send SMS messages (D)",
        "Your messages : edit SMS or MMS (D)",
        "Your messages : read SMS or MMS (D)",
        "Your messages : receive SMS (D)",
        "Your messages : receive MMS (D)",
        "Your messages : send SMS-received broadcast (S)"
    ],
    "CallLogs": [],
    "Bluetooth": [
        "Network communication : create Bluetooth connections (D)",
        "System tools : bluetooth administration (D)"
    ],
    "Calendar": [
        "Your personal information : add or modify calendar events and send email to guests (D)",
        "Your personal information : read calendar events (D)"
    ],
    "PhoneState": [
        "Default : read phone state and identity (S)",
        "Phone calls : read phone state and identity (D)"
    ],
    "Internet": [
        "Default : full Internet access (S)",
        "Network communication : full Internet access (D)"
    ],
    "ReadPhoneNumbers": []
}

PERMISSION_NAMES = list(PERMISSION_COLUMN_MAP.keys())

missing_report = {}
for perm, cols in PERMISSION_COLUMN_MAP.items():
    missing = [c for c in cols if c not in df.columns]
    if missing:
        missing_report[perm] = missing

if missing_report:
    print("WARNING -- missing raw columns detected (these are NOT silently fabricated):")
    for k, v in missing_report.items():
        print(f"   {k}: {v}")
else:
    print("All mapped raw permission columns that were expected to exist are present.")

empty_maps = [p for p, c in PERMISSION_COLUMN_MAP.items() if len(c) == 0]
print()
print("Permissions with NO raw column mapping available in this dataset")
print("(these will be constant 0 for every app -- reported honestly, not fabricated):")
print(empty_maps)

All mapped raw permission columns that were expected to exist are present.

Permissions with NO raw column mapping available in this dataset
(these will be constant 0 for every app -- reported honestly, not fabricated):
['CallLogs', 'ReadPhoneNumbers']


## 6. Category Filtering

In [6]:
if "BroadCategory" not in df.columns:
    def map_broad_category(cat):
        return "Social & Communication" if cat in CATEGORY_RAW_VALUES else "Other"

    broad_category = df["Category"].apply(map_broad_category)
    df = df.assign(BroadCategory=broad_category)
    print(f"Created BroadCategory column. Raw values mapped into '{CATEGORY}': {CATEGORY_RAW_VALUES}")

print(df["BroadCategory"].value_counts())

cat_df = df[df["BroadCategory"] == CATEGORY].copy()
print(f"\nApps in category '{CATEGORY}':", len(cat_df))
print("Benign  (Class=0):", (cat_df['Class'] == 0).sum())
print("Malicious (Class=1):", (cat_df['Class'] == 1).sum())

Created BroadCategory column. Raw values mapped into 'Social & Communication': ['Communication', 'Social']
BroadCategory
Other                     28497
Social & Communication     1502
Name: count, dtype: int64

Apps in category 'Social & Communication': 1502
Benign  (Class=0): 645
Malicious (Class=1): 857


C:\Users\Dhruvi\AppData\Local\Temp\ipykernel_9896\2814472726.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(BroadCategory=broad_category)


## 7. Permission Matrix

In [7]:
def build_permission_matrix(source_df, colmap):
    out = pd.DataFrame(index=source_df.index)
    for perm, cols in colmap.items():
        present_cols = [c for c in cols if c in source_df.columns]
        if len(present_cols) == 0:
            out[perm] = 0
        else:
            out[perm] = (source_df[present_cols].fillna(0).astype(float).sum(axis=1) > 0).astype(int)
    return out

perm_feats = build_permission_matrix(cat_df, PERMISSION_COLUMN_MAP)

permission_matrix = pd.DataFrame({
    "App": cat_df["App"].values,
    "Category": cat_df["Category"].values,
})
for p in PERMISSION_NAMES:
    permission_matrix[p] = perm_feats[p].values

# Real per-app metadata, added for the malicious-risk classifier (Section 12).
for m in METADATA_FEATURES:
    permission_matrix[m] = pd.to_numeric(cat_df[m], errors="coerce").fillna(0).values

permission_matrix["LabelEncoded"] = cat_df["Class"].values

print("permission_matrix shape:", permission_matrix.shape)
print()
print("Class distribution within category:")
print(permission_matrix["LabelEncoded"].value_counts())
print()
print("Permission counts (number of apps requesting each permission):")
print(permission_matrix[PERMISSION_NAMES].sum())
print()
print("Permission support (fraction of apps requesting each permission):")
print(permission_matrix[PERMISSION_NAMES].mean().round(4))

permission_matrix.head()

permission_matrix shape: (1502, 20)

Class distribution within category:
LabelEncoded
1    857
0    645
Name: count, dtype: int64

Permission counts (number of apps requesting each permission):
Camera               223
Location             531
Contacts             652
Microphone           288
Storage              798
SMS                  381
CallLogs               0
Bluetooth            142
Calendar              33
PhoneState           797
Internet            1291
ReadPhoneNumbers       0
dtype: int64

Permission support (fraction of apps requesting each permission):
Camera              0.1485
Location            0.3535
Contacts            0.4341
Microphone          0.1917
Storage             0.5313
SMS                 0.2537
CallLogs            0.0000
Bluetooth           0.0945
Calendar            0.0220
PhoneState          0.5306
Internet            0.8595
ReadPhoneNumbers    0.0000
dtype: float64


,App,Category,Camera,Location,Contacts,Microphone,Storage,SMS,CallLogs,Bluetooth,Calendar,PhoneState,Internet,ReadPhoneNumbers,Rating,Number of ratings,Price,Dangerous permissions count,Safe permissions count,LabelEncoded
0,"Chat, Games, Fun",Social,0,1,1,0,1,0,0,0,0,1,1,0,4.1,22987,0.0,7.0,2,1
1,Caller Intelligence App,Communication,0,0,1,0,0,1,0,0,0,1,1,0,3.3,21,0.0,9.0,1,1
2,RocketDial Language Pack,Communication,0,0,0,0,0,0,0,0,0,0,0,0,4.4,197,0.0,0.0,0,1
3,LED Dance!,Communication,0,1,0,0,0,0,0,0,0,1,1,0,4.2,94,0.0,4.0,3,0
4,Bluetooth T.K,Communication,0,0,0,0,0,0,0,1,0,0,0,0,4.0,2,0.0,2.0,0,1


In [8]:
# ---------------------------------------------------------------
# Engineered metadata features (real, derived from existing columns -- nothing fabricated)
# ---------------------------------------------------------------
permission_matrix["PermissionCount"] = permission_matrix[PERMISSION_NAMES].sum(axis=1)

_dangerous = permission_matrix["Dangerous permissions count"].astype(float)
_safe = permission_matrix["Safe permissions count"].astype(float)
permission_matrix["DangerousToSafeRatio"] = (_dangerous / (_dangerous + _safe + 1e-6)).clip(0, 1)

ENGINEERED_METADATA_FEATURES = ["PermissionCount", "DangerousToSafeRatio"]
METADATA_FEATURES = METADATA_FEATURES + ENGINEERED_METADATA_FEATURES

print("Added engineered metadata features:", ENGINEERED_METADATA_FEATURES)
print("METADATA_FEATURES now:", METADATA_FEATURES)

# Computed here (not in Section 12) because the SMOTE step (Section 8b) and the RL environment
# (Section 15) both need to know the classifier's full permission feature set too, and both run
# before Section 12 in the notebook. Value never changes once permission_matrix is built.
CLF_PERMISSION_FEATURES = [p for p in PERMISSION_NAMES if permission_matrix[p].nunique() > 1]
print("\nCLF_PERMISSION_FEATURES (non-constant permissions, used by the classifier in Section 12):")
print(CLF_PERMISSION_FEATURES)

permission_matrix[["App", "PermissionCount", "DangerousToSafeRatio"]].describe()


Added engineered metadata features: ['PermissionCount', 'DangerousToSafeRatio']
METADATA_FEATURES now: ['Rating', 'Number of ratings', 'Price', 'Dangerous permissions count', 'Safe permissions count', 'PermissionCount', 'DangerousToSafeRatio']

CLF_PERMISSION_FEATURES (non-constant permissions, used by the classifier in Section 12):
['Camera', 'Location', 'Contacts', 'Microphone', 'Storage', 'SMS', 'Bluetooth', 'Calendar', 'PhoneState', 'Internet']


,PermissionCount,DangerousToSafeRatio
count,1502.000000,1502.000000
mean,3.419441,0.687987
std,2.111444,0.244653
min,0.000000,0.000000
25%,2.000000,0.615385
50%,3.000000,0.714286
75%,5.000000,0.823529
max,9.000000,1.000000


## 8. Train / Validation / Test Split (70 / 15 / 15, Stratified)

**This is the most important cell in the notebook for scientific validity.**
The split happens *before* any weight learning, rule mining, utility/risk estimation, or RL
training. From this point on, `test_df` is only touched once, in Section 20 (Final Test Evaluation).

`train_df` -> association rule mining, malicious-risk classifier, utility estimation, RL training.
`val_df` -> reward-coefficient selection, model/threshold selection.
`test_df` -> final evaluation ONLY.

In [9]:
train_df, temp_df = train_test_split(
    permission_matrix, test_size=0.30, stratify=permission_matrix["LabelEncoded"],
    random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["LabelEncoded"],
    random_state=RANDOM_STATE
)

print("Train:", train_df.shape, "  malicious rate:", round(train_df['LabelEncoded'].mean(), 3))
print("Val:  ", val_df.shape,   "  malicious rate:", round(val_df['LabelEncoded'].mean(), 3))
print("Test: ", test_df.shape,  "  malicious rate:", round(test_df['LabelEncoded'].mean(), 3))

assert set(train_df.index).isdisjoint(set(val_df.index))
assert set(train_df.index).isdisjoint(set(test_df.index))
assert set(val_df.index).isdisjoint(set(test_df.index))
print("\nVerified: train / validation / test indices are mutually disjoint.")

Train: (1051, 22)   malicious rate: 0.571
Val:   (225, 22)   malicious rate: 0.569
Test:  (226, 22)   malicious rate: 0.571

Verified: train / validation / test indices are mutually disjoint.


# SMOTE

In [10]:
top_permissions = weights_df.head(TOP_K)["Permission"].tolist()
RL_FEATURES = top_permissions  # RL action space stays permission-only, unaffected by SMOTE/metadata

# v2: widened to CLF_PERMISSION_FEATURES (superset of RL_FEATURES) so every downstream consumer
# of train_df_balanced -- RL training AND the classifier-context lookups in make_requested_vector --
# has every permission column it needs, not just the RL's TOP_K action-space columns.
smote_permission_cols = sorted(set(CLF_PERMISSION_FEATURES) | set(RL_FEATURES))
smote_input_cols = smote_permission_cols + METADATA_FEATURES
X_train_raw = train_df[smote_input_cols].values
y_train_raw = train_df["LabelEncoded"].values

print("Before SMOTE -- training class distribution:")
print(pd.Series(y_train_raw).value_counts())

minority_count = pd.Series(y_train_raw).value_counts().min()
majority_count = pd.Series(y_train_raw).value_counts().max()
target_minority = int(majority_count * SMOTE_TARGET_RATIO)

if target_minority > minority_count:
    smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy={
        pd.Series(y_train_raw).value_counts().idxmin(): target_minority
    })
    X_res, y_res = smote.fit_resample(X_train_raw, y_train_raw)
else:
    print("Minority class already meets target ratio -- skipping SMOTE.")
    X_res, y_res = X_train_raw, y_train_raw

train_df_balanced = pd.DataFrame(X_res, columns=smote_input_cols)
train_df_balanced["LabelEncoded"] = y_res

# Round binary permission columns back to {0, 1} -- SMOTE interpolates, permissions are discrete.
# v2: rounds every permission column carried through SMOTE, not just RL_FEATURES.
for p in smote_permission_cols:
    train_df_balanced[p] = (train_df_balanced[p] >= 0.5).astype(int)

# Synthetic rows get placeholder identifiers (never real apps, never used for anything
# except training signal).
n_synthetic = len(train_df_balanced) - len(train_df)
train_df_balanced["App"] = (
    list(train_df["App"].values) + [f"SMOTE_synthetic_{i}" for i in range(n_synthetic)]
)
train_df_balanced["Category"] = CATEGORY

print("\nAfter SMOTE -- training class distribution:")
print(train_df_balanced["LabelEncoded"].value_counts())
print(f"\nTraining set size: {len(train_df)} -> {len(train_df_balanced)} ({n_synthetic} synthetic rows added)")


Before SMOTE -- training class distribution:
1    600
0    451
Name: count, dtype: int64

After SMOTE -- training class distribution:
LabelEncoded
1    600
0    600
Name: count, dtype: int64

Training set size: 1051 -> 1200 (149 synthetic rows added)


In [11]:
print(top_permissions)

['Location', 'Internet', 'Storage', 'Camera', 'PhoneState', 'Microphone', 'Bluetooth', 'Calendar', 'Contacts', 'SMS', 'CallLogs', 'ReadPhoneNumbers']


## 10. Association Rule Mining (TRAINING SET ONLY)

In [12]:
def mine_pairwise_rules(train_data, permissions, min_support, min_confidence, min_lift):
    n = len(train_data)
    rules = []
    for p1, p2 in itertools.combinations(permissions, 2):
        antecedent_mask = (train_data[p1] == 1) & (train_data[p2] == 1)
        antecedent_count = int(antecedent_mask.sum())
        support = antecedent_count / n
        if support < min_support or antecedent_count == 0:
            continue
        subset = train_data[antecedent_mask]
        for cls, cls_name in [(1, "Malicious"), (0, "Benign")]:
            confidence = (subset["LabelEncoded"] == cls).sum() / len(subset)
            class_base_rate = (train_data["LabelEncoded"] == cls).mean()
            lift = confidence / class_base_rate if class_base_rate > 0 else np.nan
            if confidence >= min_confidence and lift >= min_lift:
                rules.append({
                    "Antecedent1": p1, "Antecedent2": p2, "Consequent": cls_name,
                    "Support": support, "Confidence": confidence, "Lift": lift,
                    "AntecedentCount": antecedent_count,
                })
    return pd.DataFrame(rules)

# v2 fix: this previously passed a hardcoded 0.65 here instead of the configured
# MIN_CONFIDENCE (0.70), silently diverging from the printed/documented threshold below.
# Now uses MIN_CONFIDENCE consistently, so what's printed is what's actually applied.
association_rules_df = mine_pairwise_rules(train_df, weights_df["Permission"], MIN_SUPPORT, MIN_CONFIDENCE, MIN_LIFT)

print(f"Association rules discovered at thresholds "
      f"(support>={MIN_SUPPORT}, confidence>={MIN_CONFIDENCE}, lift>={MIN_LIFT}): {len(association_rules_df)}")

if len(association_rules_df) == 0:
    print()
    print("HONEST REPORT: No 2-permission rule among the top-K permissions reaches the")
    print("configured confidence/lift thresholds on the training data. This is a real")
    print("finding about this category/dataset, not a bug -- thresholds are NOT lowered")
    print("artificially to force rules to appear. Downstream interaction risk will be 0")
    print("for all apps as a direct, honest consequence.")
else:
    display(association_rules_df.sort_values("Lift", ascending=False))


Association rules discovered at thresholds (support>=0.04, confidence>=0.6, lift>=1.02): 9


,Antecedent1,Antecedent2,Consequent,Support,Confidence,Lift,AntecedentCount
8,Microphone,SMS,Malicious,0.074215,0.666667,1.167778,78
2,Storage,Camera,Malicious,0.120837,0.653543,1.144790,127
1,Internet,Camera,Malicious,0.140818,0.648649,1.136216,148
5,Camera,Contacts,Malicious,0.088487,0.623656,1.092437,93
4,Camera,Microphone,Malicious,0.078021,0.621951,1.089451,82
3,Camera,PhoneState,Malicious,0.110371,0.620690,1.087241,116
0,Location,Camera,Malicious,0.086584,0.615385,1.077949,91
7,Microphone,Bluetooth,Malicious,0.051380,0.611111,1.070463,54
6,PhoneState,Bluetooth,Malicious,0.062797,0.606061,1.061616,66


## 11. Association Rule Evidence

In [15]:
if len(association_rules_df) > 0:
    association_rules_df["RulePermissionWeight"] = association_rules_df.apply(
        lambda r: weight_map.get(r["Antecedent1"], 0.0) + weight_map.get(r["Antecedent2"], 0.0), axis=1
    )
    association_rules_df["EvidenceStrength"] = (
        association_rules_df["Confidence"] * np.log1p(association_rules_df["Lift"])
    )
    association_rules_df["RuleEvidence"] = (
        association_rules_df["EvidenceStrength"] * association_rules_df["RulePermissionWeight"]
    )
    max_evidence = association_rules_df["RuleEvidence"].max()
    association_rules_df["RuleEvidenceNorm"] = (
        association_rules_df["RuleEvidence"] / max_evidence if max_evidence > 0 else 0.0
    )
else:
    association_rules_df = pd.DataFrame(columns=[
        "Antecedent1", "Antecedent2", "Consequent", "Support", "Confidence", "Lift",
        "AntecedentCount", "RulePermissionWeight", "EvidenceStrength", "RuleEvidence", "RuleEvidenceNorm"
    ])

malicious_rules_df = association_rules_df[association_rules_df["Consequent"] == "Malicious"].copy()
benign_rules_df = association_rules_df[association_rules_df["Consequent"] == "Benign"].copy()

print("Malicious rules retained:", len(malicious_rules_df))
print("Benign rules retained:", len(benign_rules_df))
malicious_rules_df

Malicious rules retained: 9
Benign rules retained: 0


,Antecedent1,Antecedent2,Consequent,Support,Confidence,Lift,AntecedentCount,RulePermissionWeight,EvidenceStrength,RuleEvidence,RuleEvidenceNorm
0,Location,Camera,Malicious,0.086584,0.615385,1.077949,91,0.205623,0.450081,0.092547,0.907830
1,Internet,Camera,Malicious,0.140818,0.648649,1.136216,148,0.205623,0.492348,0.101238,0.993084
2,Storage,Camera,Malicious,0.120837,0.653543,1.144790,127,0.204425,0.498681,0.101943,1.000000
3,Camera,PhoneState,Malicious,0.110371,0.620690,1.087241,116,0.200268,0.456730,0.091468,0.897251
4,Camera,Microphone,Malicious,0.078021,0.621951,1.089451,82,0.197611,0.458317,0.090569,0.888425
5,Camera,Contacts,Malicious,0.088487,0.623656,1.092437,93,0.180239,0.460464,0.082994,0.814120
6,PhoneState,Bluetooth,Malicious,0.062797,0.606061,1.061616,66,0.188251,0.438479,0.082544,0.809708
7,Microphone,Bluetooth,Malicious,0.051380,0.611111,1.070463,54,0.185594,0.444750,0.082543,0.809700
8,Microphone,SMS,Malicious,0.074215,0.666667,1.167778,78,0.177431,0.515802,0.091519,0.897752


In [22]:
# Lookup used by the RL environment: frozenset({perm1, perm2}) -> normalized evidence.
# Only malicious rules contribute an interaction-risk penalty (benign rules are informational).
RULE_LOOKUP = {}
for _, r in malicious_rules_df.iterrows():
    RULE_LOOKUP[frozenset([r["Antecedent1"], r["Antecedent2"]])] = r["RuleEvidenceNorm"]

print("Rules feeding the RL interaction-risk term:", len(RULE_LOOKUP))

Rules feeding the RL interaction-risk term: 9


## 12. Malicious-Risk Classifier (TRAINING SET ONLY, cross-validated selection)


In [17]:
# Classifier feature set: raw permission columns (CLF_PERMISSION_FEATURES, computed in
# Section 7b) + metadata. Decoupled on purpose from RL_FEATURES/TOP_K -- see Section 12 markdown.
CLF_FEATURES = CLF_PERMISSION_FEATURES + METADATA_FEATURES

print(f"Classifier feature set: {len(CLF_PERMISSION_FEATURES)} raw permission columns "
      f"+ {len(METADATA_FEATURES)} metadata columns = {len(CLF_FEATURES)} total")
print(f"RL_FEATURES (TOP_{TOP_K}, action space only, unaffected by the above):", RL_FEATURES)

X_trainfull, y_trainfull = train_df[CLF_FEATURES].values, train_df["LabelEncoded"].values
X_val, y_val = val_df[CLF_FEATURES].values, val_df["LabelEncoded"].values
X_test, y_test = test_df[CLF_FEATURES].values, test_df["LabelEncoded"].values

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

candidate_pipelines = {
    "RandomForest": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "GradientBoosting": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    "HistGradientBoosting": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)),
    ]),
}

# Kept intentionally modest so this cell runs in reasonable time; widen if you have time to spare.
param_grids = {
    "RandomForest": {
        "clf__n_estimators": [200, 400],
        "clf__max_depth": [6, 10, None],
        "clf__min_samples_leaf": [1, 3, 5],
    },
    "GradientBoosting": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [2, 3, 4],
        "clf__learning_rate": [0.03, 0.1],
    },
    "HistGradientBoosting": {
        "clf__max_iter": [150, 300],
        "clf__max_depth": [None, 4, 8],
        "clf__learning_rate": [0.03, 0.1],
    },
    "LogisticRegression": {
        "clf__C": [0.05, 0.3, 1.0, 5.0],
    },
}

risk_model_metrics = []
fitted_grids = {}

for name, pipe in candidate_pipelines.items():
    grid = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv, n_jobs=-1, refit=True)
    grid.fit(X_trainfull, y_trainfull)
    fitted_grids[name] = grid

    val_pred = grid.predict(X_val)
    risk_model_metrics.append({
        "Model": name,
        "CV_Accuracy": grid.best_score_,
        "Val_Accuracy": accuracy_score(y_val, val_pred),
        "Val_Precision": precision_score(y_val, val_pred, zero_division=0),
        "Val_Recall": recall_score(y_val, val_pred, zero_division=0),
        "Val_F1": f1_score(y_val, val_pred, zero_division=0),
        "BestParams": grid.best_params_,
    })

risk_model_metrics_df = pd.DataFrame(risk_model_metrics).sort_values("CV_Accuracy", ascending=False).reset_index(drop=True)
print("\nCandidate malicious-risk models -- selected by cross-validated TRAIN accuracy,")
print("reported alongside an independent VALIDATION check (never used for selection):")
display(risk_model_metrics_df)

BEST_RISK_MODEL_NAME = risk_model_metrics_df.iloc[0]["Model"]
risk_model = fitted_grids[BEST_RISK_MODEL_NAME].best_estimator_  # already refit on all of X_trainfull
print(f"\nSelected risk model (by cross-validated train accuracy): {BEST_RISK_MODEL_NAME}")
print(f"Best hyperparameters: {risk_model_metrics_df.iloc[0]['BestParams']}")

# If this model exposes feature importances, show the top raw permission/metadata columns --
# a useful honest check that the extra raw columns are actually being used, not dead weight.
try:
    importances = risk_model.named_steps["clf"].feature_importances_
    top_idx = np.argsort(importances)[::-1][:15]
    print("\nTop 15 features by importance (this model only, informational):")
    for idx in top_idx:
        print(f"   {CLF_FEATURES[idx]:70s} {importances[idx]:.4f}")
except (AttributeError, KeyError):
    print("\n(Selected model does not expose feature_importances_ -- skipping importance printout.)")


def predict_malicious_proba(feature_vector_2d):
    """feature_vector_2d: array of shape (n, len(CLF_FEATURES)) -> P(Malicious) per row."""
    return risk_model.predict_proba(feature_vector_2d)[:, 1]


Classifier feature set: 10 raw permission columns + 7 metadata columns = 17 total
RL_FEATURES (TOP_12, action space only, unaffected by the above): ['Location', 'Internet', 'Storage', 'Camera', 'PhoneState', 'Microphone', 'Bluetooth', 'Calendar', 'Contacts', 'SMS', 'CallLogs', 'ReadPhoneNumbers']

Candidate malicious-risk models -- selected by cross-validated TRAIN accuracy,
reported alongside an independent VALIDATION check (never used for selection):


,Model,CV_Accuracy,Val_Accuracy,Val_Precision,Val_Recall,Val_F1,BestParams
0,RandomForest,0.715522,0.724444,0.794643,0.695312,0.741667,"{'clf__max_depth': None, 'clf__min_samples_lea..."
1,GradientBoosting,0.697445,0.733333,0.809091,0.695312,0.747899,"{'clf__learning_rate': 0.1, 'clf__max_depth': ..."
2,HistGradientBoosting,0.679364,0.688889,0.730159,0.718750,0.724409,"{'clf__learning_rate': 0.1, 'clf__max_depth': ..."
3,LogisticRegression,0.586089,0.657778,0.762887,0.578125,0.657778,{'clf__C': 0.05}



Selected risk model (by cross-validated train accuracy): RandomForest
Best hyperparameters: {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 200}

Top 15 features by importance (this model only, informational):
   Number of ratings                                                      0.4333
   Rating                                                                 0.1334
   DangerousToSafeRatio                                                   0.0803
   Dangerous permissions count                                            0.0638
   Safe permissions count                                                 0.0472
   PermissionCount                                                        0.0456
   Price                                                                  0.0410
   Storage                                                                0.0229
   Contacts                                                               0.0221
   Location                       